In [7]:
import pandas as pd

df_main    = pd.read_csv('airbnb_cleaningdata.csv')
df_bool    = pd.read_csv('data_with_bool_features.csv')
df_reviews = pd.read_csv('rentals_reviews_with_sentiment.csv')

print("=== MAIN ===")
print(df_main.columns.tolist())

print("\n=== BOOL ===")
print(df_bool.columns.tolist())

print("\n=== REVIEWS ===")
print(df_reviews.columns.tolist())

=== MAIN ===
['id', 'title', 'url', 'thumbnail', 'lat', 'lng', 'rating_overall', 'reviews_count', 'rating_accuracy', 'rating_cleanliness', 'rating_value', 'rating_location', 'price_total', 'price_original', 'discount_amount', 'description', 'has_rating', 'bedrooms', 'bathrooms']

=== BOOL ===
['id', 'title', 'url', 'thumbnail', 'lat', 'lng', 'rating_overall', 'reviews_count', 'rating_accuracy', 'rating_cleanliness', 'rating_value', 'rating_location', 'price_total', 'price_original', 'discount_amount', 'description', 'has_rating', 'description_length', 'has_wifi', 'has_pool', 'has_pyramid_view', 'view', 'view_class']

=== REVIEWS ===
['room_id', 'rating', 'comments', 'sentiment_score', 'sentiment_type']


In [8]:
import pandas as pd

df_main    = pd.read_csv('airbnb_cleaningdata.csv')
df_bool    = pd.read_csv('data_with_bool_features.csv')
df_reviews = pd.read_csv('rentals_reviews_with_sentiment.csv')

# ── Step 1: Take only the NEW columns from df_bool ──────────────────
bool_extra = df_bool[['id', 'description_length', 'has_wifi', 
                       'has_pool', 'has_pyramid_view', 'view', 'view_class']]

# ── Step 2: Aggregate reviews → one row per listing ─────────────────
df_sentiment = df_reviews.groupby('room_id').agg(
    avg_sentiment = ('sentiment_score', 'mean'),
    review_count  = ('comments', 'count')
).reset_index()

# ── Step 3: Merge everything ─────────────────────────────────────────
df = df_main.merge(bool_extra, on='id', how='left')
df = df.merge(df_sentiment, left_on='id', right_on='room_id', how='left')

In [9]:
print(df.columns.tolist())

['id', 'title', 'url', 'thumbnail', 'lat', 'lng', 'rating_overall', 'reviews_count', 'rating_accuracy', 'rating_cleanliness', 'rating_value', 'rating_location', 'price_total', 'price_original', 'discount_amount', 'description', 'has_rating', 'bedrooms', 'bathrooms', 'description_length', 'has_wifi', 'has_pool', 'has_pyramid_view', 'view', 'view_class', 'room_id', 'avg_sentiment', 'review_count']


In [10]:
print(df['discount_amount'].value_counts())

discount_amount
0.00       526
600.00       6
1000.00      5
2500.00      5
312.00       5
          ... 
201.50       1
311.20       1
120.75       1
82.50        1
330.00       1
Name: count, Length: 232, dtype: int64


In [13]:
# ── Step 4: Drop useless columns ─────────────────────────────────────
df.drop(columns=[
    'title',           # just a name, not useful for ML
    'url',             # link, not useful
    'thumbnail',       # image link, not useful
    'description',     # raw text, already extracted what we need from it
    'has_rating',      # redundant, we have rating_overall
    'room_id',         # duplicate of id after merge
], inplace=True)

# ── Step 5: Check result ──────────────────────────────────────────────
print(df.columns.tolist())
print(df.shape)
print(df.head())

['id', 'lat', 'lng', 'rating_overall', 'reviews_count', 'rating_accuracy', 'rating_cleanliness', 'rating_value', 'rating_location', 'price_original', 'bedrooms', 'bathrooms', 'description_length', 'has_wifi', 'has_pool', 'has_pyramid_view', 'view', 'view_class', 'avg_sentiment', 'review_count']
(831, 20)
                    id        lat        lng  rating_overall  reviews_count  \
0  1292713234154945394  29.973873  31.146603            4.95          133.0   
1  1508718511630646313  29.986300  31.143100            5.00           62.0   
2  1297327219631789358  29.978100  31.145400            4.91          176.0   
3  1606001853199411128  29.979090  31.146920            4.92           12.0   
4  1314833467096489875  29.978787  31.144191            4.71          125.0   

   rating_accuracy  rating_cleanliness  rating_value  rating_location  \
0             4.96                4.92          4.92             4.79   
1             5.00                5.00          4.97             4.85   


In [11]:
df.drop(columns=['price_total', 'discount_amount'], inplace=True)

# Final split
Y = df['price_original']
X = df.drop(columns=['price_original', 'id'])

In [14]:
df.columns.tolist()

['id',
 'lat',
 'lng',
 'rating_overall',
 'reviews_count',
 'rating_accuracy',
 'rating_cleanliness',
 'rating_value',
 'rating_location',
 'price_original',
 'bedrooms',
 'bathrooms',
 'description_length',
 'has_wifi',
 'has_pool',
 'has_pyramid_view',
 'view',
 'view_class',
 'avg_sentiment',
 'review_count']